# Stage 2 - Blocking (Candidate Generation)

This notebook contains:
1. **Environment & Paths**: Verify working directories, imports and the blocking configuration
2. **Design Evidence**: Training-data facts the blocking design relies on (country partition, matches per source, Indic scripts)
3. **Blocking Normalization**: Transliteration, legal/filler tokens, leet digits and phonetic keys on real records
4. **Blocking Keys**: The feature views each retrieval pass indexes, shown on a real matched pair
5. **Validation Run**: Val-split S1 queries vs the FULL train S2/S3 (~10.3M records), scored against ground truth
6. **Budget Trade-off**: Recall / F0.5 ceiling vs candidates kept per source
7. **Candidate Inspection**: Ranked candidates for sample queries, true matches marked
8. **Rescoring Weights (optional)**: Refit the logistic-regression ranking weights
9. **Training Candidates**: Scored candidate pairs for a 200k train sample (Stage 3 training data)
10. **Test Candidates**: `output/candidate_pairs.tsv` for all test S1 entities + submission-rule validation

Prerequisites (notebook `01_preprocessing.ipynb`): Stage 1 output in `dataset/preprocessed/` and the validation split in `dataset/val_split/`.
Heavy steps (val ~9 min, train ~14 min, test ~40 min on 22 threads) are skipped when their outputs already exist; set `RERUN = True` to regenerate.

In [1]:
# 1. Environment & Path Setup
import os
import sys
import json
import random
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from src.blocking import BlockingConfig, evaluate_candidates, format_report

DATA_DIR = "../student_resource/dataset"
PRE_DIR = "../dataset/preprocessed"
VAL_GT = "../dataset/val_split/val_ground_truth.tsv"
CAND_DIR = "../dataset/candidates"
OUT_DIR = "../output"
RERUN = False  # True = regenerate val/train/test candidates even if outputs exist

print(f"Python Version : {sys.version.split()[0]}")
print(f"Pandas Version : {pd.__version__}")
print(f"Root Directory : {ROOT_DIR}")
print(f"Preprocessed   : {sorted(os.listdir(PRE_DIR))}")
print(f"Val GT present : {os.path.exists(VAL_GT)}")

cfg = BlockingConfig()
print("\nRetrieval passes:")
for p in cfg.passes:
    print(f"  {p.name:<10} k={p.k:<3} max_df={p.max_df:<6} views={p.views}")
print(f"Rescoring weights : {cfg.rescore_weights}")
print(f"Budget            : {cfg.max_candidates_per_source} candidates per source (S2, S3)")

Python Version : 3.12.10
Pandas Version : 3.0.1
Root Directory : C:\Users\agarw\Desktop\PROJECTS\AMAZON_ML_CHALLENGE\amazon_hackathon
Preprocessed   : ['test_source1.parquet', 'test_source2.parquet', 'test_source3.parquet', 'train_source1.parquet', 'train_source2.parquet', 'train_source3.parquet']
Val GT present : True

Retrieval passes:
  name       k=50  max_df=1000   views={'name_word': 1.0, 'name_pair': 1.0, 'name_pkey_pair': 1.0, 'name_compact': 0.5}
  addr       k=50  max_df=1000   views={'addr_word': 1.0, 'addr_bigram': 1.0}
  name_addr  k=50  max_df=1000   views={'name_addr': 1.0, 'name_pair': 0.5, 'addr_bigram': 0.5}
Rescoring weights : {'name_char': 2.81, 'name_phon': 4.33, 'addr_word': 8.45, 'addr_bigram': 2.14, 'name_addr': 2.95}
Budget            : 25 candidates per source (S2, S3)


### 2. Design Evidence from the Training Data
- **Country partition is lossless**: if no true pair crosses country labels, S1 queries only need to be compared with S2/S3 records of the same country. The label is treated as an open set (France appears only in test).
- **Matches per source** bound the useful candidate budget.
- **Indic scripts**: S1 is ASCII, but S2/S3 India names are often in Devanagari, Tamil, Telugu, ... (handled by Stage 1 transliteration).

In [2]:
# 2. Country consistency and match counts over the full ground truth
gt = pd.read_csv(f"{DATA_DIR}/train/train_ground_truth.tsv", sep="\t", dtype=str, keep_default_na=False)
pairs = gt.assign(candidate=gt["matched_entity_ids"].str.split(",")).explode("candidate")
pairs = pairs[pairs["candidate"] != ""]

country = pd.concat([
    pd.read_parquet(f"{PRE_DIR}/train_source{n}.parquet", columns=["entity_id", "country"]) for n in (1, 2, 3)
]).set_index("entity_id")["country"]
cross = (pairs["source1_entity_id"].map(country) != pairs["candidate"].map(country)).sum()

per_source = pairs.assign(source=pairs["candidate"].str[:2]).groupby(["source1_entity_id", "source"]).size()
print(f"True pairs                     : {len(pairs):,}")
print(f"Pairs crossing country labels  : {cross:,}")
print(f"Singleton S1 entities          : {(gt['matched_entity_ids'] == '').mean():.2%}")
print(f"Matches per S1 (mean)          : {len(pairs) / len(gt):.2f}")
print("Max matches per S1 per source  :", per_source.groupby(level="source").max().to_dict())
del country

True pairs                     : 7,638,365
Pairs crossing country labels  : 0
Singleton S1 entities          : 5.58%
Matches per S1 (mean)          : 3.46


Max matches per S1 per source  : {'S2': 5, 'S3': 6}


In [3]:
# Share of non-Latin (Indic) names per source / country (first 300k rows of each raw file)
def indic_share(path, n=300_000):
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, nrows=n)
    is_indic = df["business_name"].map(lambda s: any(0x0900 <= ord(c) <= 0x0D7F for c in s))
    return is_indic.groupby(df["country"]).mean().round(4).to_dict()

for name in ("train_source1", "train_source2", "train_source3"):
    print(f"{name}: {indic_share(f'{DATA_DIR}/train/{name}.tsv')}")

train_source1: {'India': 0.0, 'US': 0.0}


train_source2: {'India': 0.2311, 'US': 0.0}


train_source3: {'India': 0.1318, 'US': 0.0}


### 3. Blocking Normalization on Real Records
Stage 1 (`clean_text`) transliterates Indic scripts before de-accenting. On top of it, blocking (`src/blocking/normalize.py`):
- drops legal forms / filler anywhere in the name (`Ltd. Ontime Pvt. Services`, `M/s`, `D.B.A.`),
- maps digits inside words back to letters (`C0astal`, `5ervices`),
- builds a phonetic skeleton that survives transliteration (`kulopal pichinas` ~ `global business`).

In [4]:
# 3. Transliteration + blocking normalization on random Indic-script S2 names
from src.preprocessing import clean_text
from src.blocking.normalize import name_tokens, phonetic_name

s2_head = pd.read_csv(f"{DATA_DIR}/train/train_source2.tsv", sep="\t", dtype=str, keep_default_na=False, nrows=200_000)
indic = s2_head[s2_head["business_name"].map(lambda s: any(0x0900 <= ord(c) <= 0x0D7F for c in s))]

for raw in indic["business_name"].sample(8, random_state=random.randint(0, 10_000)):
    toks = name_tokens(clean_text(raw))
    print(f"{raw}\n    clean: {clean_text(raw)}\n    keys : {toks}  phonetic: {phonetic_name(toks)}")

print("\nNoise examples:")
for raw in ["Ltd. Ontime Pvt. Services", "M/s ashutoshbrosprivate.com", "C0astal Tungsten, LLC",
            "Platinum Commercial 5érvices Corp", "Anselma's Hóldings (ID: [14307)]"]:
    print(f"  {raw:<40} -> {name_tokens(clean_text(raw))}")

ಸೂಪರ್ ಹೆಲ್ತ್‌ಕೇರ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್
    clean: supar heltker praivet limited
    keys : ['supar', 'heltker']  phonetic: spr altkr
हाई कंसल्टेंट्स प्राइवेट लिमिटेड
    clean: hai kansaltents praivet limited
    keys : ['hai', 'kansaltents']  phonetic: a knsltnts
સેવન ટેક્નોલોજી પ્રાઇવેટ લિમિટેડ
    clean: sevan teknoloji praivet limited
    keys : ['sevan', 'teknoloji']  phonetic: svn tknlj
ജയ് ഡെവലപ്പേഴ്സ് പ്രൈവറ്റ് ലിമിറ്റഡ്
    clean: jay devalappezhs praivatt limittad
    keys : ['jay', 'devalappezhs', 'praivatt']  phonetic: j tvlpss prvt
ग्रीन यूनिवर्सल हॉस्पिटैलिटी प्रा. लि.
    clean: grin yunivarsal hospitailiti pra li
    keys : ['grin', 'yunivarsal', 'hospitailiti', 'pra', 'li']  phonetic: krn anvrsl asptlt pr l
बालाजी सॉफ्टवेयर प्राइवेट लिमिटेड
    clean: balaji sophtaveyar praivet limited
    keys : ['balaji', 'sophtaveyar']  phonetic: plj sptvr
கோல்டு சொல்யூஷன்ஸ் பிரைவேட் லிமிடெட்
    clean: koltu cholyushans piraivet limitet
    keys : ['koltu', 'cholyushans']  phonetic: k

### 4. Blocking Keys (Feature Views)
Single name words are common because the corpus reuses a small business vocabulary. Word **pairs**, address **bigrams** and **name x address** cross keys are rare and identifying. Retrieval indexes only keys with document frequency <= `max_df` (1,000), which bounds the cost per query.

In [5]:
# 4. Keys of a real matched pair from the validation split, and the keys they share
from src.blocking.features import record_features

val_gt = pd.read_csv(VAL_GT, sep="\t", dtype=str, keep_default_na=False)
val_pairs = val_gt.assign(candidate=val_gt["matched_entity_ids"].str.split(",")).explode("candidate")
val_pairs = val_pairs[val_pairs["candidate"] != ""]
s1_id, t_id = val_pairs.sample(1, random_state=random.randint(0, 10_000)).iloc[0][["source1_entity_id", "candidate"]]

def lookup(entity_id):
    path = f"{PRE_DIR}/train_source{entity_id[1]}.parquet"
    return pd.read_parquet(path, filters=[("entity_id", "==", entity_id)]).iloc[0]

a, b = lookup(s1_id), lookup(t_id)
print(f"S1: {a.clean_name} | {a.clean_address}")
print(f"{t_id[:2]}: {b.clean_name} | {b.clean_address}\n")
for view in dict.fromkeys(v for p in cfg.passes for v in p.views):
    fa, fb = record_features(a.clean_name, a.clean_address, view), record_features(b.clean_name, b.clean_address, view)
    print(f"{view:<15} shared={sorted(set(fa) & set(fb))[:6]}  (S1 has {len(fa)} keys)")

S1: kolkata career pvt ltd | 103 4a nabalia para road kolkata kolkata howrah west bengal
S3: kolkata career pvt ltd | 103 4a calcutta howrah wb

name_word       shared=['career', 'kolkata']  (S1 has 2 keys)
name_pair       shared=['career|kolkata']  (S1 has 1 keys)
name_pkey_pair  shared=['klkt|krr']  (S1 has 1 keys)
name_compact    shared=['kolkatac']  (S1 has 1 keys)
addr_word       shared=['103', '4', '4a', 'howrah']  (S1 has 11 keys)
addr_bigram     shared=['103_4a', '4a_4']  (S1 has 10 keys)
name_addr       shared=['career|103', 'career|4', 'career|4a', 'career|howrah', 'kolkata|103', 'kolkata|4']  (S1 has 20 keys)


### 5. Validation Run
The 10k validation S1 queries are searched against the **full** train S2/S3, so candidate-set sizes and recall reflect the real 1.7M x 10M problem (a small distractor pool would make blocking look better than it is).

In [6]:
# 5. Blocking on the validation split (~9 min); skipped if the report already exists
val_report_path = f"{CAND_DIR}/val_blocking_report.json"
if RERUN or not os.path.exists(val_report_path):
    !python ../scripts/run_blocking.py --split val --preprocessed-dir {PRE_DIR} --val-gt {VAL_GT} --out-dir {CAND_DIR}
else:
    print(f"Using existing {val_report_path} (set RERUN = True to regenerate)")

val_report = json.load(open(val_report_path))
print(f"\nRun time: {val_report['seconds'] / 60:.1f} min | step seconds: {val_report['step_seconds']}\n")
print(format_report(val_report["metrics"]))

Using existing ../dataset/candidates/val_blocking_report.json (set RERUN = True to regenerate)

Run time: 9.0 min | step seconds: {'addr': 28.3, 'name': 10.2, 'name_addr': 26.4}

queries             : 10,000 (560 singletons)
true pairs          : 34,632
candidate pairs     : 500,000  (mean 50.0 / median 50 / p95 50 / max 50 per query, 0 queries with none)
pair recall         : 0.9777   by source {'S2': 0.97939, 'S3': 0.97609}
F0.5 ceiling        : 0.9925
queries full recall : 0.9319   zero recall: 0.0023
reduction ratio     : 0.99999516
recall by country   : {'india': 0.96575, 'us': 0.98576}
recall@k/source     : 1:0.3666, 2:0.6337, 3:0.7987, 5:0.9379, 10:0.9745, 15:0.9763, 20:0.9772, 25:0.9777, 30:0.9777, 40:0.9777, 50:0.9777
  pass name        : recall 0.7341  unique 0.0044
  pass addr        : recall 0.8765  unique 0.0269
  pass name_addr   : recall 0.9457  unique 0.0091


In [7]:
# 6. Budget trade-off: recall and F0.5 ceiling if fewer candidates per source are kept
val_pairs_scored = pd.read_parquet(f"{CAND_DIR}/val_candidate_pairs.parquet")
rows = []
for budget in (1, 3, 5, 10, 15, 20, 25):
    r = evaluate_candidates(val_pairs_scored[val_pairs_scored["source_rank"] < budget], val_gt)
    rows.append({"per_source": budget, "per_S1": 2 * budget, "pair_recall": r["pair_recall"],
                 "f05_ceiling": r["f05_ceiling"], "zero_recall_queries": r["query_zero_recall"]})
pd.DataFrame(rows).round(4)

,per_source,per_S1,pair_recall,f05_ceiling,zero_recall_queries
0,1,2,0.3666,0.6828,0.1624
1,3,6,0.7987,0.9314,0.0226
2,5,10,0.9379,0.9819,0.0041
3,10,20,0.9745,0.9916,0.0024
4,15,30,0.9763,0.9921,0.0023
5,20,40,0.9772,0.9924,0.0023
6,25,50,0.9777,0.9925,0.0023


In [8]:
# 7. Ranked candidates for a random validation query (true matches marked)
qid = random.choice(val_gt.loc[val_gt["matched_entity_ids"] != "", "source1_entity_id"].tolist())
truth = set(val_gt.set_index("source1_entity_id").loc[qid, "matched_entity_ids"].split(","))
cands = val_pairs_scored[val_pairs_scored["source1_entity_id"] == qid].sort_values("candidate_score", ascending=False)

ids = cands["candidate_entity_id"].tolist()
records = pd.concat([
    pd.read_parquet(f"{PRE_DIR}/train_source{n}.parquet", filters=[("entity_id", "in", ids)]) for n in (2, 3)
]).set_index("entity_id")

q = lookup(qid)
print(f"Query {qid}: {q.clean_name} | {q.clean_address}  ({len(truth)} true matches)")
view = cands.assign(
    true=cands["candidate_entity_id"].isin(truth),
    name=cands["candidate_entity_id"].map(records["clean_name"]),
    address=cands["candidate_entity_id"].map(records["clean_address"]),
)[["candidate_entity_id", "true", "source_rank", "candidate_score", "name", "address"]]
view.head(12)

Query S1-966052534: continental highland services llc | 20607 39th drive phoenix az  (5 true matches)


,candidate_entity_id,true,source_rank,candidate_score,name,address
469750,S2-972563811,True,0,20.680000,continental highland services,20607 39th drive phoenix az
469751,S2-514444805,True,1,20.680000,llc c0ntinental highland services,20607 39th drive phoenix az
469752,S3-31272469,True,0,20.679998,continental highland services llc,
469753,S2-618818415,False,2,16.814238,continental highland,
469754,S3-430898309,True,1,14.947432,continental highland llc center,20607 39th drive arrowhead arizona
469755,S2-414949357,False,3,14.043571,continental highland services group,20616 39th drive phoenix az
469756,S2-162040889,False,4,13.957692,highland continental llc services,
469757,S2-735498622,False,5,13.453390,continental corp services,
469758,S2-209343597,False,6,13.453390,continental services,
469759,S2-211848115,True,7,11.950071,continentalhighlandsvc com,39th drive phoenix az


### 8. Rescoring Weights (optional)
The retrieved union is ranked by a weighted sum of exact similarities. The weights in `src/blocking/config.py` are logistic-regression coefficients (true pair vs not) fitted on validation candidates. This cell refits them and reports held-out recall for current vs fitted weights; the result is saved as a config usable with `run_blocking.py --config`.

In [9]:
# 8. Refit rescoring weights (~10 min); off by default
REFIT = False
fitted_path = f"{CAND_DIR}/fitted_blocking_config.json"
if REFIT:
    !python ../scripts/fit_rescore_weights.py --preprocessed-dir {PRE_DIR} --gt {VAL_GT} --out {fitted_path}
if os.path.exists(fitted_path):
    print("Fitted weights :", json.load(open(fitted_path))["rescore_weights"])
print("Default weights:", cfg.rescore_weights)

Fitted weights : {'name_char': 1.507, 'name_phon': 5.257, 'addr_word': 12.509, 'addr_bigram': 0.42, 'name_addr': 6.41}
Default weights: {'name_char': 2.81, 'name_phon': 4.33, 'addr_word': 8.45, 'addr_bigram': 2.14, 'name_addr': 2.95}


### 9. Training Candidates for Stage 3
A random 200k train S1 sample (validation entities excluded) is blocked against the full train S2/S3. The scored pairs (`score_*` retrieval cosines, `sim_*` exact similarities, `candidate_score`, `source_rank`) are the Stage 3 training data. Because the rescoring weights were fitted on the validation split, these metrics are also the unbiased estimate of blocking quality.

In [10]:
# 9. Training candidates (~14 min); skipped if the report already exists
train_report_path = f"{CAND_DIR}/train_blocking_report.json"
if RERUN or not os.path.exists(train_report_path):
    !python ../scripts/run_blocking.py --split train --sample-s1 200000 --preprocessed-dir {PRE_DIR} --val-gt {VAL_GT} --train-gt {DATA_DIR}/train/train_ground_truth.tsv --out-dir {CAND_DIR}
else:
    print(f"Using existing {train_report_path} (set RERUN = True to regenerate)")

print(format_report(json.load(open(train_report_path))["metrics"]))
train_head = pd.read_parquet(f"{CAND_DIR}/train_candidate_pairs.parquet").head()
train_head.astype({c: "float32" for c in train_head.columns if str(train_head[c].dtype) == "float16"})

Using existing ../dataset/candidates/train_blocking_report.json (set RERUN = True to regenerate)
queries             : 200,000 (11,114 singletons)
true pairs          : 692,670
candidate pairs     : 10,000,000  (mean 50.0 / median 50 / p95 50 / max 50 per query, 0 queries with none)
pair recall         : 0.9759   by source {'S2': 0.97709, 'S3': 0.97487}
F0.5 ceiling        : 0.9916
queries full recall : 0.9263   zero recall: 0.0024
reduction ratio     : 0.99999516
recall by country   : {'india': 0.96091, 'us': 0.986}
recall@k/source     : 1:0.3662, 2:0.6318, 3:0.7961, 5:0.9339, 10:0.9729, 15:0.9747, 20:0.9754, 25:0.9759, 30:0.9759, 40:0.9759, 50:0.9759
  pass name        : recall 0.7350  unique 0.0049
  pass addr        : recall 0.8743  unique 0.0259
  pass name_addr   : recall 0.9445  unique 0.0096


,source1_entity_id,candidate_entity_id,source,country,candidate_score,source_rank,score_name,score_addr,score_name_addr,sim_name_char,sim_name_phon,sim_addr_word,sim_addr_bigram,sim_name_addr
0,S1-914683160,S2-17846916,S2,us,20.679998,0,0.5,0.0000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
1,S1-914683160,S2-239958886,S2,us,20.679998,1,0.5,0.0000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
2,S1-914683160,S3-861665144,S3,us,11.821659,0,0.0,0.9375,0.643066,0.575684,0.483643,0.681641,0.452148,0.468018
3,S1-914683160,S3-955128603,S3,us,8.766994,1,0.5,0.0000,0.159058,1.000000,1.000000,0.136963,0.000000,0.159058
4,S1-914683160,S2-559121160,S2,us,8.475272,2,0.0,0.0000,0.251953,1.000000,1.000000,0.070068,0.000000,0.251953


### 10. Test Candidates
All 1.73M test S1 entities (US, India and the unseen France) are blocked against the test S2/S3. The output `output/candidate_pairs.tsv` is checked against the submission rules of `student_resource/utils/validate_submission.py`, including the check that every candidate ID exists in the test S2/S3 files.

In [11]:
# 10. Test candidates (~40 min); skipped if output/candidate_pairs.tsv already exists
test_tsv = f"{OUT_DIR}/candidate_pairs.tsv"
if RERUN or not os.path.exists(test_tsv):
    !python ../scripts/run_blocking.py --split test --preprocessed-dir {PRE_DIR} --out-dir {CAND_DIR} --submission-dir {OUT_DIR}
else:
    print(f"Using existing {test_tsv} (set RERUN = True to regenerate)")

Using existing ../output/candidate_pairs.tsv (set RERUN = True to regenerate)


In [12]:
# Validate candidate_pairs.tsv with the official validator's rules (matching_results.tsv comes in Stage 5)
import importlib.util

spec = importlib.util.spec_from_file_location("validate_submission", "../student_resource/utils/validate_submission.py")
validator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validator)

test_dir = f"{DATA_DIR}/test"
required = validator.read_ids(f"{test_dir}/test_source1.tsv")
warnings, errors = [], []
valid_ids = validator.load_match_targets(test_dir, warnings)
mapping = validator.validate_id_list_file(test_tsv, validator.CANDIDATE_HEADER, "candidate_entity_ids",
                                          required, valid_ids, errors)
sizes = pd.Series({k: len(v) for k, v in mapping.items()})
print(f"Candidates per S1: mean {sizes.mean():.1f}, min {sizes.min()}, max {sizes.max()}")
print("PASS" if not errors else f"FAIL: {errors}", "| warnings:", warnings or "none")
del mapping, valid_ids

  candidate_pairs.tsv: 1732544 rows (0 empty, 1732544 non-empty).


Candidates per S1: mean 50.0, min 20, max 50
PASS | warnings: none
